<a href="https://colab.research.google.com/github/Rogusliv/MWW-gen/blob/main/microWakeWord_traine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# microWakeWord Trainer (headless, no Docker/GUI)

Runtime > Change runtime type > GPU (T4 or better) before running.

Set your wake word and options in the cell below, then Runtime > Run all.

In [ ]:
WAKE_WORD = "sample"        # spelled phonetically, e.g. "hey mycroft"
WAKE_WORD_TITLE = "sanoke"  # pretty display name

SAMPLES = 50000  #number of samples 50k is quite large but a good sample size
BATCH_SIZE = 100
TRAINING_STEPS = 40000

USE_DRIVE = False
DRIVE_SYNC_DIR = "/content/drive/MyDrive/microWakeWord_data"  # your existing Drive folder

Put your recorded wake word samples in the following folder microWakeWord_data/personal_samples/

These must be in separate files.


Formated as .wav mono 16k.

In [ ]:
import os
os.environ["LANGUAGE"] = "en"

In [ ]:
!nvidia-smi

Sat Sep 19 16:01:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')  # needed to sync personal_samples in / trained model out

if USE_DRIVE:
    DATA_DIR = DRIVE_SYNC_DIR
else:
    DATA_DIR = "/content/data"

os.makedirs(DATA_DIR, exist_ok=True)
os.environ["DATA_DIR"] = DATA_DIR
os.environ["DRIVE_SYNC_DIR"] = DRIVE_SYNC_DIR
print("DATA_DIR =", DATA_DIR)

Mounted at /content/drive
DATA_DIR = /content/data


In [ ]:
!apt-get -qq update
!apt-get -qq install -y git wget curl unzip patch ninja-build build-essential cmake pkg-config \
    ca-certificates ffmpeg sox libsox-fmt-all libsndfile1 espeak-ng python3.12-venv

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../00-libpython3.12-dev_3.12.3-1ubuntu0.17_amd64.deb ...
Unpacking libpython3.12-dev:amd64 (3.12.3-1ubuntu0.17) over (3.12.3-1ubuntu0.16) ...
Preparing to unpack .../01-libpython3.12t64_3.12.3-1ubuntu0.17_amd64.deb ...
Unpacking libpython3.12t64:amd64 (3.12.3-1ubuntu0.17) over (3.12.3-1ubuntu0.16) ...
Preparing to unpack .../02-python3.12_3.12.3-1ubuntu0.17_amd64.deb ...
Unpacking python3.12 (3.12.3-1ubuntu0.17) over (3.12.3-1ubuntu0.16) ...
Preparing to unpac

In [ ]:
![ -d /content/mww-scripts ] || git clone https://github.com/TaterTotterson/microWakeWord-Trainer-Nvidia-Docker /content/mww-scripts
!chmod -R a+x /content/mww-scripts/cli /content/mww-scripts/train_wake_word /content/mww-scripts/run.sh

Cloning into '/content/mww-scripts'...
remote: Enumerating objects: 963, done.
remote: Counting objects: 100% (439/439), done.
remote: Compressing objects: 100% (210/210), done.
remote: Total 963 (delta 309), reused 300 (delta 218), pack-reused 524 (from 2)
Receiving objects: 100% (963/963), 1.76 MiB | 9.84 MiB/s, done.
Resolving deltas: 100% (520/520), done.


In [ ]:
# Builds /content/data/.venv with GPU-flavored TensorFlow/PyTorch/onnxruntime.
# This step alone can take 15-30+ minutes; it only needs to run once per DATA_DIR.
setup_cmd = "cd '{data_dir}' && /content/mww-scripts/cli/setup_python_venv --gpu --data-dir='{data_dir}'".format(
    data_dir=os.environ["DATA_DIR"]
)
print(setup_cmd)
!bash -lc "{setup_cmd}"

cd '/content/data' && /content/mww-scripts/cli/setup_python_venv --gpu --data-dir='/content/data'
===== Setting up Python environment /content/data/.venv =====
   ===== Creating new virtualenv at '/content/data/.venv' =====
   ===== Installing common requirements =====
......................................................................................
   ===== Installing TensorFlow stack (tensorflow[and-cuda]==2.20.0) =====
.....................................................................................................................................................ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.63.1 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
scipy 1.12.0 requires numpy<1.29.0,>=1.22.4, but you have numpy 2.5.3 which is incompatible.
.
   ===== Installing torch and torchaudio [cuda] =====
........

In [ ]:
fix_cmd = (
    "source '{data_dir}/.venv/bin/activate' && "
    "pip install --upgrade 'scipy>=1.13' 'numba>=0.61'"
).format(data_dir=os.environ["DATA_DIR"])
print(fix_cmd)
!bash -lc "{fix_cmd}"

source '/content/data/.venv/bin/activate' && pip install --upgrade 'scipy>=1.13' 'numba>=0.61'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 43.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 129.1 MB/s  0:00:00
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.46.0
    Uninstalling llvmlite-0.46.0:
      Successfully uninstalled llvmlite-0.46.0
  Attempting uninstall: numba
    Found existing installation: numba 0.63.1
    Uninstalling numba-0.63.1:
      Successfully uninstalled numba-0.63.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [numba]


In [ ]:
import os

check_script = """
import numpy, scipy, numba
print('numpy', numpy.__version__)
print('scipy', scipy.__version__)
print('numba', numba.__version__)
import tensorflow as tf
print('tf', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
"""

with open("/content/verify_check.py", "w") as f:
    f.write(check_script)

venv_python = f"{os.environ['DATA_DIR']}/.venv/bin/python"
!{venv_python} /content/verify_check.py

numpy 2.3.5
scipy 1.18.1
numba 0.67.0
2026-09-19 16:08:20.027446: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-19 16:08:20.097339: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-19 16:08:21.750798: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
tf 2.20.0
GPU: [PhysicalDevice(name='/phys

In [ ]:
import os, shutil

drive_personal = os.path.join(os.environ["DRIVE_SYNC_DIR"], "personal_samples")
local_personal = os.path.join(os.environ["DATA_DIR"], "personal_samples")

if os.environ["DATA_DIR"] != os.environ["DRIVE_SYNC_DIR"]:
    if os.path.isdir(drive_personal):
        os.makedirs(local_personal, exist_ok=True)
        for fname in os.listdir(drive_personal):
            src = os.path.join(drive_personal, fname)
            if os.path.isfile(src):  # skip .ipynb_checkpoints and other stray folders
                shutil.copy2(src, os.path.join(local_personal, fname))
        print(f"Synced {len(os.listdir(local_personal))} personal sample(s) from Drive.")
    else:
        print(f"No personal_samples folder found at {drive_personal} (create it on Drive and re-run this cell if you have recordings to add).")
else:
    print("DATA_DIR is Drive itself; nothing to sync.")

Synced 11 personal sample(s) from Drive.


In [15]:
# Downloads the negative/background training datasets (AudioSet, FMA, CHiME,
# MIT RIRs, curated negative sets).
dataset_cmd = (
    "cd '{data_dir}' && source '{data_dir}/.venv/bin/activate' && "
    "/content/mww-scripts/cli/setup_training_datasets "
    "--cleanup-archives=true --cleanup-intermediate-files=true "
    "--data-dir='{data_dir}'"
).format(data_dir=os.environ["DATA_DIR"])
print(dataset_cmd)
!bash -lc "{dataset_cmd}"

cd '/content/data' && source '/content/data/.venv/bin/activate' && /content/mww-scripts/cli/setup_training_datasets --cleanup-archives=true --cleanup-intermediate-files=true --data-dir='/content/data'

===== Setting up Training Datasets =====

===== Checking negative datasets: dinner_party dinner_party_eval no_speech speech =====
   Unzipping dinner_party.zip
   Cleaning up dinner_party.zip
   Unzipping dinner_party_eval.zip
   Cleaning up dinner_party_eval.zip
   Unzipping no_speech.zip
   Cleaning up no_speech.zip
   Unzipping speech.zip
   Cleaning up speech.zip
   Negative datasets complete
===== Checking MIT environmental RIRs =====
   Found 270 MIT environmental RIR files on Hugging Face mirror
   MIT RIR download progress: 1/270 files (1 downloaded, 0 reused)
   MIT RIR download progress: 25/270 files (25 downloaded, 0 reused)
   MIT RIR download progress: 50/270 files (50 downloaded, 0 reused)
   MIT RIR download progress: 75/270 files (75 downloaded, 0 reused)
   MIT RIR downl

In [16]:
# Quick check of how much space DATA_DIR is actually using so far.
!du -sh "$DATA_DIR"/* 2>/dev/null | sort -rh

39G	/content/data/training_datasets
201M	/content/data/tools
808K	/content/data/personal_samples
12K	/content/data/cuda


In [ ]:
cmd = (
    "source '{data_dir}/.venv/bin/activate' && "
    "/content/mww-scripts/train_wake_word "
    "--samples={samples} --batch-size={batch_size} --training-steps={steps} "
    "--data-dir='{data_dir}' '{wake_word}' '{title}'"
).format(
    data_dir=os.environ["DATA_DIR"],
    samples=SAMPLES,
    batch_size=BATCH_SIZE,
    steps=TRAINING_STEPS,
    wake_word=WAKE_WORD,
    title=WAKE_WORD_TITLE,
)
print(cmd)
!bash -lc "{cmd}"

source '/content/data/.venv/bin/activate' && /content/mww-scripts/train_wake_word --samples=50000 --batch-size=100 --training-steps=40000 --data-dir='/content/data' 'asmodius' 'asmodeus'
===== Running 'asmodius(asmodeus)' generation, augmentation and training =====
                 GPU Name: NVIDIA A100-SXM4-40GB
       Compute Capability:     8.0
Streaming Multiprocessors:     108
        CUDA Cores per SM:      64
         Total CUDA Cores:    6912
             Total Memory:   40441 mb
              Free Memory:   40013 mb

===== Generating 50000 wake-word samples (language=en, accent=mixed, tts=hybrid) =====
→ Waiting for the TTS GPU lock
===== Direct TTS corpus plan (hybrid, en) =====
   English accent emphasis: mixed
   omnivoice: 12500 sample(s)
   qwen3: 12500 sample(s)
   moss: 12500 sample(s)
   piper: 12500 sample(s)
   safety duration: 0.46–2.37s; static, silence, clipping, rambling, and exact duplicates are rejected
→ /content/mww-scripts/cli/setup_modern_tts_envs --engine=

In [ ]:
import shutil

zip_path = shutil.make_archive("/content/trained_wake_word", "zip", f"{os.environ['DATA_DIR']}/trained_wake_words")
print(f"Zipped to: {zip_path}")

from google.colab import files
files.download(zip_path)